In [142]:
import sys 

print(sys.executable)
print(sys.version)

c:\Projects\FlightRisk\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


phase 2,here i will continue localy and work on several point including :
local reproducibility,
validate raw data,
define modeling population,
prepare leakage-safe X and y

In [143]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn 
import kagglehub 
from pathlib import Path

print ("numpy version : ", np.__version__)
print ("pandas version : ", pd.__version__)
print ("matplotlib version : ", plt.matplotlib.__version__)
print ("sklearn version : ", sklearn.__version__)
print ("kagglehub version : ", kagglehub.__version__)
Path.cwd()

numpy version :  2.5.3
pandas version :  3.0.5
matplotlib version :  3.11.1
sklearn version :  1.9.0
kagglehub version :  1.0.2


WindowsPath('c:/Projects/FlightRisk/notebooks')

In [144]:
sample_path = Path("../data/raw/flight_data_2024_sample.csv")
data_dictionary_path = Path("../data/raw/flight_data_2024_data_dictionary.csv")

sample_path.exists()
data_dictionary_path.exists()

flights = pd.read_csv(sample_path)
flights.shape
flights.head()

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,"Dallas/Fort Worth, TX",Texas,...,0,151.0,144.0,119.0,835.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,AA,148.0,CLT,"Charlotte, NC",North Carolina,...,0,286.0,273.0,253.0,1773.0,0,0,0,0,0
2,2024,12,12,4,2024-12-12,9E,5440.0,CHA,"Chattanooga, TN",Tennessee,...,0,59.0,50.0,29.0,106.0,0,0,0,0,0
3,2024,4,8,1,2024-04-08,WN,1971.0,OMA,"Omaha, NE",Nebraska,...,0,180.0,177.0,163.0,1099.0,0,0,0,0,0
4,2024,2,16,5,2024-02-16,WN,862.0,BWI,"Baltimore, MD",Maryland,...,0,90.0,96.0,76.0,399.0,0,0,0,0,0


phase 1


In [145]:
#create a copy of mine data frame to work with
model_data = flights.copy()
print(model_data.shape)
#filtring the dataframe from cancelled and diverted flights
model_data = model_data[
    (model_data["cancelled"] == 0) 
    & (model_data["diverted"] == 0)  
    & (model_data["arr_delay"].notna())
    ]
print(model_data.shape)
#using assert to confirm that canclled and diverted flights are exluded from the dataframe
assert (model_data["cancelled"] == 0).all() ,"canclled values must be 0 for this dataset"
assert (model_data["diverted"] == 0).all() ,"diverted values must be 0 for this dataset"
assert (model_data["arr_delay"].notna()).all() , "NaN arr_delay values must be exluded from dataframe"

(10000, 35)
(9836, 35)


inspecting duplicates shemas and identifiers 

In [146]:
#inspecting duplicates and unique combinations identifiers 
print(model_data.duplicated().value_counts())
print(model_data["op_carrier_fl_num"].nunique())
candidate_key = model_data.duplicated(subset=["op_carrier_fl_num", "fl_date" , "op_unique_carrier", "origin"])
legs = model_data[model_data.duplicated(subset=["op_carrier_fl_num", "fl_date" , "op_unique_carrier"], keep=False)]
print(legs[["op_carrier_fl_num", "fl_date" , "op_unique_carrier", "origin", "dest", "crs_dep_time"]])

assert candidate_key.sum() == 0, "this unique candidate key combinition must produce no duplicates"

False    9836
Name: count, dtype: int64
4562
      op_carrier_fl_num     fl_date op_unique_carrier origin dest  \
400              5038.0  2024-01-03                9E    DTW  SDF   
597              1922.0  2024-08-05                AA    PIT  ORD   
1731              369.0  2024-07-21                WN    BWI  MCO   
2243             2683.0  2024-02-22                UA    PIT  ORD   
2490             5038.0  2024-01-03                9E    SDF  DTW   
3364              369.0  2024-07-21                WN    DEN  RIC   
3813             2683.0  2024-02-22                UA    ORD  SAN   
4239             5463.0  2024-05-26                OH    RIC  CLT   
4436             5463.0  2024-05-26                OH    CLT  RIC   
6386             1153.0  2024-09-06                DL    ATL  RSW   
6943             1922.0  2024-08-05                AA    ORD  PIT   
7061             3453.0  2024-09-23                MQ    DFW  FSM   
7458             3453.0  2024-09-23                MQ    F

Creating a three sepereate variables list including potenial features provided before departure ,
model target (arr_delay),
leakage 